# Masterclass: Tool Engineering & Enterprise Tool Architecture in LangChain & LangGraph

Welcome to the updated masterclass on **Tool Engineering**. In modern Agentic AI, **Tools** provide LLMs with dynamic execution capabilities—enabling them to query search engines, interact with databases, compute mathematical operations, trigger external APIs, or execute protocol-level RPCs via the **Model Context Protocol (MCP)**.

### Core Architectural Distinction: Graph Nodes vs. Agent Tools

- **Graph Node**: A **deterministic workflow orchestration step** controlled by the LangGraph engine (e.g., input validation, audit logging, guardrails).
- **Agent Tool**: A **callable capability** exposed to an LLM via a name, description, and input schema. The LLM dynamically decides **if, when, and with what arguments** to invoke the tool.

```text
                                  USER QUERY
                                      │
                                      ▼
                             [LLM Agent Node]
                                      │
              ┌───────────────────────┼───────────────────────┐
              ▼                       ▼                       ▼
      [Calculator Tool]        [Web Search Tool]       [Retriever Tool]
              └───────────────────────┬───────────────────────┘
                                      │
                                      ▼
                             [Validator Node] (Deterministic Guardrail)
                                      │
                                      ▼
                             [Final Output Node]
```

### Comprehensive Tool Creation Methods & Master Priority Matrix

| Priority | Tool Creation Method | Primary Use Case |
|---|---|---|
| ⭐⭐⭐⭐⭐ | `@tool` | Standard Python functions converted to tools |
| ⭐⭐⭐⭐⭐ | `@tool + Pydantic` | Strict type validation & complex structured inputs |
| ⭐⭐⭐⭐ | Async `@tool` (`async def`) | Non-blocking API/DB network tools |
| ⭐⭐⭐⭐ | `create_retriever_tool()` | RAG Document Retrievers |
| ⭐⭐⭐⭐ | Prebuilt Integrations (`TavilySearch`) | Ecosystem integration tools |
| ⭐⭐⭐⭐ | Model Context Protocol (MCP) | Remote microservices & enterprise tool servers |
| ⭐⭐⭐ | `ToolRuntime` | Reading LangGraph graph state dynamically inside tools |
| ⭐⭐ | `StructuredTool.from_function()` | Multi-parameter functions constructed imperatively |
| ⭐⭐ | Subclassing `BaseTool` | Enterprise object-oriented tool creation |
| ⭐ | `Tool(...)` / `from_function()` | Simple single-string input tool wrappers |

## 1. Setup & Environment Verification
Load environment variables from `.env` and verify API key availability.

In [7]:
# Verify setup and check availability of API keys in environment
from dotenv import load_dotenv
import os
load_dotenv()

print("Setup loaded.")
print("GROQ_API_KEY available:", bool(os.getenv("GROQ_API_KEY")))
print("GOOGLE_API_KEY available:", bool(os.getenv("GOOGLE_API_KEY")))
print("TAVILY_API_KEY available:", bool(os.getenv("TAVILY_API_KEY")))

Setup loaded.
GROQ_API_KEY available: True
GOOGLE_API_KEY available: True
TAVILY_API_KEY available: True


## 2. Tool Creation via `@tool` Decorator

The `@tool` decorator is the primary method to convert standard Python functions into LangChain/LangGraph tools. LangChain automatically inspects function signatures, type annotations, and docstrings to build the tool metadata.

### 2.1 Basic `@tool` Decorator & Attribute Inspection
Inspect automatically inferred tool attributes: `.name`, `.description`, and `.args`.

In [11]:
# Import @tool decorator from langchain_core.tools
from langchain_core.tools import tool

In [3]:
# Define basic calculator function with @tool decorator
@tool
def add_basic(a: int, b:int) -> int:
    """Add two numbers"""
    return a + b

In [4]:
# Inspect automatically inferred tool name
print("Name:", add_basic.name)

Name: add_basic


In [5]:
# Inspect automatically extracted tool description from docstring
print("Description:", add_basic.description)

Description: Add two numbers


In [6]:
# Inspect inferred tool input arguments and types
print("Args:", add_basic.args)

Args: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [7]:
# Invoke tool using dictionary input format
add_basic.invoke({"a": 20, "b": 30})

50

In [8]:
# Execute tool invocation and store/print result
result = add_basic.invoke({"a": 20, "b": 30})
print("Execution result:", result)

Execution result: 50


### 2.2 Custom Tool Names & Execution Flags
Customize tool metadata using `@tool("custom_name", description=..., return_direct=True)`. Setting `return_direct=True` returns tool results directly to the user without passing them back through the LLM.

In [10]:
# Define tool with custom name override
@tool("calculator")
def add_with_custom_name(a:int, b:int) -> int:
    """Add two numbers."""
    return a + b

In [11]:
# Verify custom tool name and execute invocation
print("Tool name:", add_with_custom_name.name)
print("Execution result:", add_with_custom_name.invoke({"a": 10, "b": 15}))

Tool name: calculator
Execution result: 25


In [12]:
# Define tool with explicit name, description, and return_direct flag
@tool(
    "multiply_numbers",
    description="Multiply two integers and return the result.",
    return_direct=False,
)
def multiply_with_options(a, b):
    return a * b

In [13]:
# Inspect custom tool attributes and execute invocation
print("Name:", multiply_with_options.name)
print("Description:", multiply_with_options.description)
print("return_direct:", multiply_with_options.return_direct)
print("Execution result:", multiply_with_options.invoke({"a": 6, "b": 7}))

Name: multiply_numbers
Description: Multiply two integers and return the result.
return_direct: False
Execution result: 42


In [22]:
# we get result beacuse we have not enforced a schema
print("Execution result:", multiply_with_options.invoke({"a": "areeb", "b": 7}))

Execution result: areebareebareebareebareebareebareeb


### 2.3 Strict Input Validation with Pydantic `args_schema` 
Use a Pydantic `BaseModel` to enforce strict type checking, parameter bounds, and field descriptions. Invalid inputs raise a `ValidationError` before bad parameters reach function logic.

In [16]:
# Import Pydantic BaseModel and Field for strict schema enforcement
from pydantic import BaseModel, Field, ValidationError

In [17]:
# Define Pydantic schema for input argument validation
class CalculatorInputTest(BaseModel):
    a: int = Field(description="First integer")
    b: int = Field(description="Second integer")

In [18]:
# Bind Pydantic args_schema to tool
@tool(args_schema=CalculatorInputTest)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [19]:
# Inspect generated Pydantic JSON schema
print("Schema:", multiply.args_schema.model_json_schema())

Schema: {'properties': {'a': {'description': 'First integer', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second integer', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'CalculatorInputTest', 'type': 'object'}


In [20]:
# Execute tool with valid integer arguments
print("Execution result:", multiply.invoke({"a": 8, "b": 9}))

Execution result: 72


In [21]:
# it wont work with string as input as we have implemented enforce schema
multiply.invoke({"a": "areeb", "b": 9})

ValidationError: 1 validation error for CalculatorInputTest
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='areeb', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

### 2.4 Automatic Schema Generation with `@tool(parse_docstring=True)`
Automatically extract parameter descriptions from Google/NumPy style docstring `Args:` blocks without writing manual Pydantic schema classes.

In [23]:
# Define tool with parse_docstring=True to auto-generate schema from docstring Args:
@tool(parse_docstring=True)
def search_with_docstring(query: str, limit: int) -> str:
    """Search documents.

    Args:
        query: Search query entered by the user.
        limit: Maximum number of results.
    """
    return f"Searching for '{query}' with limit={limit}"

In [24]:
# Inspect auto-generated JSON schema from docstring
print("Args schema:")
print(search_with_docstring.args_schema.model_json_schema())

Args schema:
{'description': 'Search documents.', 'properties': {'query': {'description': 'Search query entered by the user.', 'title': 'Query', 'type': 'string'}, 'limit': {'description': 'Maximum number of results.', 'title': 'Limit', 'type': 'integer'}}, 'required': ['query', 'limit'], 'title': 'search_with_docstring', 'type': 'object'}


In [25]:
# Execute invocation on docstring-parsed tool
print("Execution result:", search_with_docstring.invoke({"query": "LangGraph memory","limit": 3,}))

Execution result: Searching for 'LangGraph memory' with limit=3


### 2.5 Asynchronous Tools (`async def` & `ainvoke`)
Define non-blocking async tools using `async def` and execute them asynchronously via `await tool.ainvoke(...)`.

In [26]:
# Import asyncio for asynchronous execution
import asyncio

In [28]:
# Define async tool function using async def
@tool
async def get_data_async(url: str) -> str:
    """Fetch data asynchronously"""
    await asyncio.sleep(0.1)
    return f"Data from {url}"

In [29]:
# Execute async tool using await tool.ainvoke()
result = await get_data_async.ainvoke({"url": "https://example.com"})
print("Async execution result:", result)

Async execution result: Data from https://example.com


## 3. Accessing LangGraph State inside Tools via `ToolRuntime` 

In LangGraph applications, tools often require access to the active **graph state** (e.g., user context, conversation history, user ID). Passing `ToolRuntime` as a tool parameter allows tools to dynamically read state at runtime.

In [30]:
# Import dependencies for ToolRuntime in LangGraph
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, ToolRuntime
from langchain_core.messages import AIMessage

In [31]:
# Define custom graph state inheriting from MessagesState
class RuntimeState(MessagesState):
    question: str

In [32]:
# Define tool accepting ToolRuntime parameter to read active graph state
@tool
def read_question_from_runtime(runtime: ToolRuntime) -> str:
    """Read the current question from LangGraph state."""
    return runtime.state["question"]

In [33]:
# Node function creating synthetic AIMessage tool call for demonstration
def create_demo_tool_call(state: RuntimeState):
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "read_question_from_runtime",
                    "args": {},
                    "id": "demo_call_1",
                    "type": "tool_call",
                }],
            )
        ]
    }

In [34]:
# Instantiate StateGraph using RuntimeState
runtime_builder = StateGraph(RuntimeState)

In [35]:
# Register nodes and connect execution edges
runtime_builder.add_node("create_call", create_demo_tool_call)
runtime_builder.add_node("tools", ToolNode([read_question_from_runtime]))
runtime_builder.add_edge(START, "create_call")
runtime_builder.add_edge("create_call", "tools")
runtime_builder.add_edge("tools", END)

In [36]:
# Compile RuntimeState graph workflow
runtime_graph = runtime_builder.compile()

In [37]:
# Invoke graph with initial state
runtime_result = runtime_graph.invoke({
    "question": "What is LangGraph?",
    "messages": [],
})

In [38]:
# Print tool result retrieved from runtime graph state
print("Tool result:", runtime_result["messages"][-1].content)

Tool result: What is LangGraph?


## 4. Imperative Tool Construction (`Tool` & `StructuredTool`)

Tools can be constructed imperatively using `Tool(...)` for single-string inputs or `StructuredTool.from_function(...)` for multi-parameter functions.

In [39]:
# Import Tool class from langchain_core.tools
from langchain_core.tools import Tool

In [40]:
# Define simple single-argument search function
def simple_search_function(query: str) -> str:
    return f"Searching for {query}"

In [41]:
# Construct Tool instance imperatively
simple_search_tool = Tool(
    name="simple_search",
    func=simple_search_function,
    description="Search for information.",
)

In [42]:
# Inspect Tool attributes and invoke with single string argument
print("Name:", simple_search_tool.name)
print("Execution result:", simple_search_tool.invoke("LangGraph"))

Name: simple_search
Execution result: Searching for LangGraph


### 4.1 Single-Input Tools via `Tool.from_function()`

In [43]:
# Define search function for Tool.from_function()
def search_from_function(query: str) -> str:
    return f"Result for {query}"

In [44]:
# Demonstrate Tool.from_function construction
Tool.from_function(
    func=search_from_function,
    name="search_from_function",
    description="Search information.",
)

Tool(name='search_from_function', description='Search information.', func=<function search_from_function at 0x0000019A538B5940>)

In [45]:
# Store Tool.from_function instance in variable
from_function_tool = Tool.from_function(
    func=search_from_function,
    name="search_from_function",
    description="Search information.",
)

In [46]:
# Invoke tool created via Tool.from_function()
print("Execution result:", from_function_tool.invoke("Agentic AI"))

Execution result: Result for Agentic AI


### 4.2 Multi-Input Tools via `StructuredTool` 
Use `StructuredTool.from_function()` or `StructuredTool(args_schema=...)` for functions accepting multiple parameters.

In [47]:
# Import StructuredTool and construct tool for multi-argument function
from langchain_core.tools import StructuredTool

def calculate_tax_test(income: float, tax_rate: float) -> float:
    return income * tax_rate

tax_tool_test = StructuredTool.from_function(
    func=calculate_tax_test,
    name="calculate_tax",
    description="Calculate tax from income and tax rate.",
)

print("Args:", tax_tool_test.args)
print("Execution result:", tax_tool_test.invoke({
    "income": 100000,
    "tax_rate": 0.20,
}))

Args: {'income': {'title': 'Income', 'type': 'number'}, 'tax_rate': {'title': 'Tax Rate', 'type': 'number'}}
Execution result: 20000.0


In [48]:
# Define Pydantic schema and construct StructuredTool with explicit schema
class MultiplyInputTest2(BaseModel):
    a: int = Field(description="First number")
    b: int = Field(description="Second number")
    a: int = Field(description="First number")
    b: int = Field(description="Second number")

def direct_multiply(a: int, b: int) -> int:
    return a * b

direct_structured_tool = StructuredTool(
    name="direct_multiply",
    description="Multiply two numbers.",
    func=direct_multiply,
    args_schema=MultiplyInputTest2,
)

print("Execution result:", direct_structured_tool.invoke({
    "a": 12,
    "b": 4,
}))

Execution result: 48


## 5. Enterprise Subclassing of `BaseTool` 

For enterprise applications, subclassing `BaseTool` offers object-oriented encapsulation, custom initialization, state management, and explicit `_run` and `_arun` implementations.

In [49]:
# Import BaseTool and Type from typing & langchain_core.tools
from typing import Type
from langchain_core.tools import BaseTool

In [50]:
# Define Pydantic input schema for custom BaseTool subclass
class SearchInputTest(BaseModel):
    query: str = Field(description="Search query")

In [51]:
# Subclass BaseTool to create enterprise custom tool class
class MySearchToolTest(BaseTool):
    name: str = "my_search"
    description: str = "Search my custom database."
    args_schema: Type[BaseModel] = SearchInputTest

    def _run(self, query: str) -> str:
        return f"Custom database result for: {query}"

In [52]:
# Instantiate custom BaseTool class
custom_base_tool = MySearchToolTest()

In [53]:
# Execute custom BaseTool invocation
print("Execution result:", custom_base_tool.invoke({
    "query": "LangGraph state management"
}))

Execution result: Custom database result for: LangGraph state management


## 6. Converting Retrievers into Tools (`create_retriever_tool`)

In Agentic RAG, document retrievers must be exposed as tools so the LLM can autonomously query internal knowledge bases when necessary.

In [1]:
# Import retriever tool dependencies
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.tools import create_retriever_tool

In [2]:
# Define demo retriever subclassing BaseRetriever
class DemoRetriever(BaseRetriever):
    def _get_relevant_documents(self, query: str, *, run_manager=None):
        return [
            Document(
                page_content=f"Demo internal document relevant to: {query}",
                metadata={"source": "demo.txt"},
            )
        ]

demo_retriever = DemoRetriever()

In [3]:
# Convert retriever into LangChain tool using create_retriever_tool()
retriever_tool_test = create_retriever_tool(
    demo_retriever,
    name="search_company_documents",
    description="Search internal company documents.",
)

In [4]:
# Invoke retriever tool on user query
print("Execution result:")
print(retriever_tool_test.invoke({
    "query": "What is the company leave policy?"
}))

Execution result:
Demo internal document relevant to: What is the company leave policy?


## 7. LLM Tool Binding (`bind_tools`) & Schema Conversions

- **Important**: `bind_tools` and `with_structured_output` are fundamental concepts for making models tool-aware and enforcing structured responses.

To make an LLM tool-aware, we use `llm.bind_tools([tool1, tool2])`. The LLM inspects tool JSON schemas and emits `AIMessage.tool_calls` when a tool invocation is required.


In [5]:
# Initialize variable for Groq model instance
groq_llm = None

In [8]:
# Initialize ChatGroq model instance if GROQ_API_KEY is available
if os.getenv("GROQ_API_KEY"):
    from langchain_groq import ChatGroq

    groq_llm = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0,
    )
    print("Groq model initialized.")
else:
    print("Skipping live Groq tests because GROQ_API_KEY is not available.")

Groq model initialized.


In [9]:
# Test baseline LLM response generation
groq_llm.invoke("hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond. The instruction: "You are ChatGPT, a large language model trained by OpenAI." There\'s no special instruction. Just respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 72, 'total_tokens': 129, 'completion_time': 0.062392433, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.005731582, 'prompt_tokens_details': None, 'queue_time': 0.310364748, 'total_time': 0.068124015}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a05940-c6ab-7420-8b42-5b04b2240187-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 57, 'total_tokens': 129, 'output_token_details': {'reasoning': 39}})

### 7.1 Testing Tool Invocation & `bind_tools()` Binding

In [12]:
# Define sample weather tool
@tool
def weather_callable(location: str) -> str:
    """Get weather for a location."""
    return f"Demo weather for {location}: 28°C"

In [13]:
# Local Python implementation works independently:
print("Direct Python execution:", weather_callable.invoke("Bangalore"))

Direct Python execution: Demo weather for Bangalore: 28°C


In [14]:
# Bind weather tool to Groq LLM model
callable_bound_model = groq_llm.bind_tools(
    [weather_callable]
)

In [15]:
# Invoke model with prompt triggering tool call
ai_message = callable_bound_model.invoke("Use the weather tool for Bangalore.")

In [16]:
# Inspect output AIMessage
ai_message

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the weather tool.', 'tool_calls': [{'id': 'fc_5be31d55-14bd-4d74-a92a-dfe6433a43cf', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'weather_callable'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 126, 'total_tokens': 159, 'completion_time': 0.050120145, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.076793772, 'prompt_tokens_details': None, 'queue_time': 0.370700015, 'total_time': 0.126913917}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d23c14756c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0594d-6534-7423-b4a5-cf33ba788760-0', tool_calls=[{'name': 'weather_callable', 'args': {'location': 'Bangalore'}, 'id': 'fc_5be31d55-14bd-4d74-a92a-dfe6433a43cf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

In [17]:
# Inspect generated tool calls JSON structure
print("Generated tool calls:")
print(ai_message.tool_calls)

Generated tool calls:
[{'name': 'weather_callable', 'args': {'location': 'Bangalore'}, 'id': 'fc_5be31d55-14bd-4d74-a92a-dfe6433a43cf', 'type': 'tool_call'}]


### 7.2 Schema Conversions with `convert_to_openai_tool` 
Inspect how LangChain converts callables, Pydantic models, TypedDicts, and raw JSON schemas into standard provider tool format.

In [ ]:
# For OpenAI
# class GetWeatherSchema(BaseModel):
#     """Get current weather."""
#     location: str = Field(description="City name")

# from langchain_core.utils.function_calling import convert_to_openai_tool

# print("Converted tool schema:")
# print(convert_to_openai_tool(GetWeatherSchema))

In [ ]:
# pydantic_bound_model = groq_llm.bind_tools(
#     [GetWeatherSchema],
#     tool_choice="GetWeatherSchema",
# )

# ai_message = pydantic_bound_model.invoke(
#     "Get the weather for Bangalore."
# )

# print("Generated tool call:")
# print(ai_message.tool_calls)

# print(
# "Important: There is no weather implementation here, "
# "so the returned tool call still needs an executor."
# )

In [ ]:
# class WeatherTypedDict(TypedDict):
#     """Get current weather."""
#     location: str

# print("Converted TypedDict tool schema:")
# print(convert_to_openai_tool(WeatherTypedDict))

In [ ]:
# if groq_llm is not None:
#     typed_dict_model = groq_llm.bind_tools(
#         [WeatherTypedDict],
#         tool_choice="WeatherTypedDict",
#     )

#     ai_message = typed_dict_model.invoke(
#         "Get the weather for New York."
#     )

#     print("Generated tool call:")
#     print(ai_message.tool_calls)

In [ ]:
# Old way of creating a tool (not required anymore)
# raw_weather_tool = {
#     "type": "function",
#     "function": {
#         "name": "get_weather_raw",
#         "description": "Get current weather",
#         "parameters": {
#             "type": "object",
#             "properties": {
#                 "location": {
#                     "type": "string",
#                     "description": "City name",
#                 }
#             },
#             "required": ["location"],
#         },
#     },
# }

# print("Raw schema:")
# print(raw_weather_tool)

In [ ]:
# if groq_llm is not None:
#     raw_bound_model = groq_llm.bind_tools(
#         [raw_weather_tool],
#         tool_choice="get_weather_raw",
#     )

#     ai_message = raw_bound_model.invoke(
#         "Get the weather for Delhi."
#     )

#     print("Generated tool call:")
#     print(ai_message.tool_calls)

## 8. Toolkits (`BaseToolkit`) & Prebuilt Integrations

A **Toolkit** aggregates related tools into a reusable factory class.

In [18]:
# Define custom BaseToolkit subclass grouping math tools
from langchain_core.tools import BaseToolkit

@tool
def toolkit_add(a: int, b: int) -> int:
    """Add numbers."""
    return a + b

@tool
def toolkit_multiply(a: int, b: int) -> int:
    """Multiply numbers."""
    return a * b

class DemoMathToolkit(BaseToolkit):
    def get_tools(self):
        return [
            toolkit_add,
            toolkit_multiply,
        ]

demo_toolkit = DemoMathToolkit()
toolkit_tools = demo_toolkit.get_tools()

print("Toolkit tools:", [t.name for t in toolkit_tools])
print("Add result:", toolkit_tools[0].invoke({"a": 2, "b": 3}))
print("Multiply result:", toolkit_tools[1].invoke({"a": 4, "b": 5}))

Toolkit tools: ['toolkit_add', 'toolkit_multiply']
Add result: 5
Multiply result: 20


### 8.1 Prebuilt Ecosystem Tools (`TavilySearch`)
Use ready-made integration tools directly in LangChain / LangGraph agents.

In [19]:
# Instantiate TavilySearch prebuilt integration tool
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=3,
    search_depth="basic",
    topic="general",
)

result = tavily_tool.invoke({
    "query": "What is LangGraph?"
})

print(result)

{'query': 'What is LangGraph?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.ibm.com/think/topics/langgraph', 'title': 'What is LangGraph?', 'content': 'LangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models (LLMs) in a scalable and efficient manner. At its core, LangGraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an AI agent workflow. [...] Agent systems: LangGraph provides a framework for building agent-based systems, which can be used in applications such as robotics, autonomous vehicles or video games.\n\nLLM applications: By using LangGraph’s capabilities, developers can build more sophisticated AI models that learn and improve over time. Norwegian Cruise Line

## 9. Executing Tool Calls in LangGraph via `ToolNode` 

`bind_tools()` only enables the LLM to emit tool call requests (`AIMessage.tool_calls`). **`ToolNode`** is the prebuilt LangGraph node that actually executes the requested tools and appends `ToolMessage` results.

In [20]:
# Import ToolNode and AIMessage for graph tool execution
from langchain_core.messages import AIMessage
from langgraph.prebuilt import ToolNode

In [21]:
# Define sample subtraction tool
@tool
def subtract(a: int, b: int) -> int:
    """Subtract b from a."""
    return a - b

In [22]:
# Instantiate ToolNode containing subtraction tool
tool_node = ToolNode([subtract])

In [23]:
# Invoke ToolNode with synthetic AIMessage tool call
tool_node_result = tool_node.invoke({
    "messages": [AIMessage(content="",
                tool_calls=[{
                "name": "subtract",
                "args": {
                    "a": 100,
                    "b": 35,
                },
                "id": "call_subtract_1",
                "type": "tool_call",
            }],
        )
    ]
})

ValueError: Missing required config key 'N/A' for 'tools'.

In [ ]:
# Inspect complete ToolNode output state
print("ToolNode result:")
print(tool_node_result)

In [ ]:
# Print extracted ToolMessage content
print("Tool output:")
print(tool_node_result["messages"][-1].content)

## 10. Model Context Protocol (MCP) Server & Adapter Integration

The **Model Context Protocol (MCP)** is an open standard for connecting AI models to secure remote tool servers. Using `langchain-mcp-adapters`, LangChain agents can seamlessly connect to external MCP servers.

#### Create an MCP tool and load it into LangChain

```bash
uv pip install -U mcp langchain-mcp-adapters
```

In [ ]:
# Import MCP server dependencies
from pathlib import Path
import sys

from mcp.server.fastmcp import FastMCP

In [ ]:
# Create FastMCP server instance
mcp = FastMCP("Math")

In [ ]:
# Define tool on FastMCP server
@mcp.tool()
def add_mcp(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [ ]:
mcp.run(transport="stdio")

### 10.1 Connecting to Single & Multi-Server MCP Clients

In [ ]:
# Create MCP server file dynamically
mcp_server_path = Path("demo_math_mcp_server.py")
mcp_server_path.write_text(mcp_server_code, encoding="utf-8")

print("Created MCP server:", mcp_server_path.resolve())

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [ ]:
# Connect MultiServerMCPClient to local FastMCP server
mcp_client = MultiServerMCPClient({
    "math": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(mcp_server_path.resolve())],
    }
})

In [ ]:
# Fetch tools from MCP server client
mcp_tools = await mcp_client.get_tools()

In [ ]:
# Print loaded MCP tools
print("Loaded MCP tools:", [t.name for t in mcp_tools])

In [ ]:
# Select add_mcp tool
add_mcp_tool = next(
        t for t in mcp_tools
        if t.name == "add_mcp"
    )

In [ ]:
# Execute remote MCP tool via ainvoke()
print(
    "MCP execution result:",
    await add_mcp_tool.ainvoke({
        "a": 10,
        "b": 25,
    })
    )

In [ ]:
# Create multiple MCP server script files
server_1 = Path("demo_mcp_add_server.py")
server_2 = Path("demo_mcp_multiply_server.py")

server_1.write_text(r'''
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("AddServer")

@mcp.tool()
def remote_add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

if __name__ == "__main__":
    mcp.run(transport="stdio")
''', encoding="utf-8")

server_2.write_text(r'''
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("MultiplyServer")

@mcp.tool()
def remote_multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")
''', encoding="utf-8")

print("Created two MCP server files.")

In [ ]:
# Connect MultiServerMCPClient to multiple remote servers
from langchain_mcp_adapters.client import MultiServerMCPClient

multi_client = MultiServerMCPClient({
    "addition": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(server_1.resolve())],
    },
    "multiplication": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(server_2.resolve())],
    },
})

multi_tools = await multi_client.get_tools()

print("Tools loaded from multiple MCP servers:")
print([t.name for t in multi_tools])

tool_map = {
    t.name: t
    for t in multi_tools
}

print(
    "remote_add:",
    await tool_map["remote_add"].ainvoke({
        "a": 5,
        "b": 6,
    })
)

print(
    "remote_multiply:",
    await tool_map["remote_multiply"].ainvoke({
        "a": 5,
        "b": 6,
    })
)

### 10.2 Bidirectional Conversion: LangChain Tools <-> FastMCP Tools

In [ ]:
# Define sample tool for FastMCP conversion
@tool
def langchain_add_for_mcp(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [ ]:
# Convert LangChain tool to FastMCP tool using to_fastmcp()
try:
    from langchain_mcp_adapters.tools import to_fastmcp

    fastmcp_tool = to_fastmcp(
        langchain_add_for_mcp
    )

    print("Converted FastMCP tool:")
    print(fastmcp_tool)

except ImportError:
    print("Install langchain-mcp-adapters to run this section.")

## 11. Architectural Decision Framework: Node vs. Tool

### 11.1 Key Conceptual Principles
- **Tool**: A callable capability exposed via Name, Description, and Schema. Python application executes the function; LLM generates arguments.
- **Node**: A deterministic workflow step in a graph pipeline (e.g. guardrails, audit logging, state transformations).

### 11.2 Decision Tree: Choosing the Right Tool Creation Method

```text
I have a Python Function
         │
         ▼
    Simple Inputs? ───(YES)───► Use @tool
         │
        (NO - Complex / Validated Inputs)
         │
         ▼
   @tool + Pydantic (args_schema)
         │
         ├─► Network / API / DB I/O? ─────(YES)───► Use async @tool
         ├─► Document Retriever? ──────────(YES)───► Use create_retriever_tool()
         ├─► Needs Graph State? ───────────(YES)───► Use ToolRuntime
         ├─► Object-Oriented Control? ─────(YES)───► Subclass BaseTool
         └─► Remote Microservice Server? ─(YES)───► Use MCP (FastMCP)
```

### 11.3 Architectural Rule of Thumb
- **Use Nodes** for mandatory processing steps that MUST execute every time (Validation, Audit, Formatting, Finalization).
- **Use Tools** for dynamic capabilities that the model should choose based on user prompt context (Search, Calculator, Weather, External APIs).